Packages import

In [157]:
import os
import re
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

Apollo Scraper

In [158]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [159]:
group_id = input("Enter group ID: ")
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [160]:
page_dom = BeautifulSoup(response.text, "html.parser")

In [161]:
group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

ZICSS1-1211


In [162]:
classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [163]:
classes = classes.loc[classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])]

In [164]:
classes[['Day','Start time', 'hyphen', 'End time', "Duration"]] = classes['Dzień, godzina'].str.split(' ',expand=True)

In [165]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [166]:
classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [167]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [168]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [169]:
classes.to_csv(f"schedules/{group}.csv")

In [170]:
classes 

,Termin,Przedmiot,Typ,Nauczyciel,Sala,Day,Start time,End time,Duration
1,2026-02-23,Computer Programming 2,ćwiczenia,dr Katarzyna Wójcik,Paw.A 013 lab.,Pn,13:15,15:45,3
4,2026-02-24,Probability and Statistics,wykład,prof. dr hab. Andrzej Sokołowski,Paw.F 008,Wt,09:45,11:15,2
6,2026-02-24,Probability and Statistics,ćwiczenia,prof. dr hab. Andrzej Sokołowski,Paw.A 07 lab.,Wt,13:15,14:45,2
8,2026-02-24,Discrete mathematics,wykład,dr Grzegorz Kosiorowski,Rakowicka 16 sala 11,Wt,15:00,16:30,2
9,2026-02-24,Business Law,wykład,dr Jacek Lachner,Rakowicka 16 sala 11,Wt,18:30,20:00,2
...,...,...,...,...,...,...,...,...,...
293,2026-06-11,Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,Paw.A 07 lab.,Cz,15:00,16:30,2
295,2026-06-11,Information Systems,ćwiczenia,mgr inż. Justyna Olczak,Paw.A 014 lab.,Cz,16:45,17:30,1
296,2026-06-11,Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,Paw.A 07 lab.,Cz,18:30,20:00,2
298,2026-06-17,Discrete mathematics,egzamin,dr Grzegorz Kosiorowski,Paw.C Nowa Aula,Śr,10:30,13:00,3
